# Stress Prediction v19 — Heterogeneous Ensemble + Rank Features

Building on v17/v18 diagnosis. Public LB history: v16=0.377, v17=0.315, v18=?. We need ≥0.45.

## Key insights from v18 diagnostic
1. **OOF underestimates final model.** Class 1 has only 3 source subjects (F1ZM/DT5C/P4DZ), with P4DZ holding 74% of class-1 data. Holding out any of these in GroupKFold removes most class-1 signal from training. The final model trains on all 7 subjects and is more class-1-aware than CV folds suggest.
2. **Raw OOF dist (16/3/80) vs raw test dist (51/8/41) showed strong covariate shift.** New subjects' sensor patterns trigger different model regions.
3. **Single-model + calibration plateaued.** v17 went too far toward minorities (LB 0.315), v18 corrected back (probably ~0.38). Calibration alone cannot recover signal the model doesn't have.

## v19 changes
- **Heterogeneous ensemble**: LightGBM + sklearn HistGradientBoosting + sklearn ExtraTrees. Different inductive biases → different errors → averaging soft probabilities reduces single-model overconfidence on unseen subjects.
- **Subject-relative percentile-rank features**: each window's sensor stat as a percentile of that subject's full sensor distribution. Unitless, fully cross-subject transferable.
- **Stronger LGBM regularization**: smaller num_leaves (15), higher min_child_samples (30), more L2.
- **Train final ensemble on all data** with 3 seeds × 3 models. CV uses GroupKFold for honest reporting.
- **Calibration tuned on OOF** to match train prior, but with awareness that final model is less class-2-biased than OOF — so we expect a smaller alpha than v18.

Compatible with stated submission rules: no external data, no transfer learning, only declared libraries (lightgbm, sklearn, pandas, numpy, scipy).


In [1]:
%pip -q install lightgbm scikit-learn pandas numpy scipy


[notice] A new release of pip is available: 26.0 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from scipy import stats as spstats

from sklearn.impute import SimpleImputer
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.preprocessing import StandardScaler

import lightgbm as lgb

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_DIR = Path('.')
TRAIN_DATA  = pd.read_csv(DATA_DIR / 'train-sensor.csv')
TRAIN_LABEL = pd.read_csv(DATA_DIR / 'train-label.csv')
TEST_DATA   = pd.read_csv(DATA_DIR / 'test-sensor.csv')
TEST_LABEL  = pd.read_csv(DATA_DIR / 'test-label.csv')

print('Raw shapes')
print('  TRAIN_DATA :', TRAIN_DATA.shape)
print('  TRAIN_LABEL:', TRAIN_LABEL.shape)
print('  TEST_DATA  :', TEST_DATA.shape)
print('  TEST_LABEL :', TEST_LABEL.shape)
print('Train PIDs:', sorted(TRAIN_LABEL['pid'].astype(str).unique()))
print('Test PIDs :', sorted(TEST_LABEL['pid'].astype(str).unique()))


Raw shapes
  TRAIN_DATA : (4694400, 8)
  TRAIN_LABEL: (815, 4)
  TEST_DATA  : (5921280, 8)
  TEST_LABEL : (1028, 4)
Train PIDs: ['43JW', 'C8Q6', 'DT5C', 'F1ZM', 'HDS9', 'P4DZ', 'TPQI']
Test PIDs : ['01Z2', '2XO3', 'D1XP', 'NQRB', 'SE4Q', 'SNG7', 'TF0Y', 'Y21H']


## Cleaning

In [3]:
SENSOR_COLS = ['accel_x', 'accel_y', 'accel_z', 'eda', 'heart_rate', 'temperature']

def clean_sensor(df):
    out = df.copy()
    out['pid'] = out['pid'].astype(str)
    out['timestamp'] = pd.to_numeric(out['timestamp'], errors='coerce').astype(float)
    for c in SENSOR_COLS:
        out[c] = pd.to_numeric(out[c], errors='coerce').astype(float)
    out['accel_x'] = out['accel_x'].clip(-128, 127)
    out['accel_y'] = out['accel_y'].clip(-128, 127)
    out['accel_z'] = out['accel_z'].clip(-128, 127)
    out['eda'] = out['eda'].clip(0, 60)
    out['heart_rate'] = out['heart_rate'].clip(40, 190)
    out['temperature'] = out['temperature'].clip(20, 40)
    return out.sort_values(['pid', 'timestamp']).reset_index(drop=True)

def clean_label(df):
    out = df.copy()
    out['id'] = pd.to_numeric(out['id'], errors='raise').astype(int)
    out['pid'] = out['pid'].astype(str)
    out['timestamp'] = pd.to_numeric(out['timestamp'], errors='coerce').astype(float)
    out['stress'] = pd.to_numeric(out['stress'], errors='coerce')
    return out

TRAIN_DATA = clean_sensor(TRAIN_DATA)
TEST_DATA = clean_sensor(TEST_DATA)
TRAIN_LABEL = clean_label(TRAIN_LABEL)
TEST_LABEL = clean_label(TEST_LABEL)
print('Cleaned. Train sensor rows:', len(TRAIN_DATA), 'Test sensor rows:', len(TEST_DATA))


Cleaned. Train sensor rows: 4694400 Test sensor rows: 5921280


## Per-Subject Standardization + Subject-Distribution Reference

Two transforms per sensor channel, computed independently per PID (no train leakage; works on test using only that test subject's own stream):

1. **Robust z-score**: `(x - subject_median) / (1.4826 * subject_MAD)`. Subject-relative magnitude.
2. **Within-subject percentile rank**: each timestamp's value mapped to its percentile (0..1) within that subject's full distribution. Unitless, scale-free, ideal for cross-subject transfer.

Plus we precompute each subject's quantile reference points (p10, p25, p50, p75, p90) from their full sensor stream, so that windowed features can express "this window's mean was above the subject's p75 baseline" without leakage.

In [4]:
def per_subject_transforms(sensor_df, sensor_cols):
    out = sensor_df.copy()
    g = out.groupby('pid')[sensor_cols]
    
    # Robust z-score
    med = g.transform('median')
    mad = g.transform(lambda x: np.median(np.abs(x - np.median(x))))
    std_fb = g.transform('std')
    mad = mad.where(mad > 0, std_fb)
    mad = mad.where(mad > 0, 1.0)
    for c in sensor_cols:
        out[f'{c}_z'] = (out[c] - med[c]) / (1.4826 * mad[c])
    
    # Within-subject percentile rank (rank divided by group size)
    for c in sensor_cols:
        out[f'{c}_pr'] = out.groupby('pid')[c].rank(pct=True, method='average')
    
    return out

# Precompute subject quantiles so window-level features can compare against per-subject baselines
def subject_quantile_ref(sensor_df, sensor_cols, qs=(0.1, 0.25, 0.5, 0.75, 0.9)):
    refs = {}
    for pid, grp in sensor_df.groupby('pid'):
        refs[pid] = {}
        for c in sensor_cols:
            vals = grp[c].dropna().values
            if len(vals) == 0:
                refs[pid][c] = {q: 0.0 for q in qs}
            else:
                refs[pid][c] = {q: float(np.quantile(vals, q)) for q in qs}
    return refs

TRAIN_DATA = per_subject_transforms(TRAIN_DATA, SENSOR_COLS)
TEST_DATA  = per_subject_transforms(TEST_DATA,  SENSOR_COLS)

TRAIN_REFS = subject_quantile_ref(TRAIN_DATA, SENSOR_COLS)
TEST_REFS  = subject_quantile_ref(TEST_DATA,  SENSOR_COLS)

Z_COLS  = [f'{c}_z'  for c in SENSOR_COLS]
PR_COLS = [f'{c}_pr' for c in SENSOR_COLS]
print('Z columns:',  Z_COLS)
print('PR columns:', PR_COLS)
print('Sample subject HR quantiles (train PID 0):')
first_pid = list(TRAIN_REFS.keys())[0]
print(f'  {first_pid}: {TRAIN_REFS[first_pid]["heart_rate"]}')


Z columns: ['accel_x_z', 'accel_y_z', 'accel_z_z', 'eda_z', 'heart_rate_z', 'temperature_z']
PR columns: ['accel_x_pr', 'accel_y_pr', 'accel_z_pr', 'eda_pr', 'heart_rate_pr', 'temperature_pr']
Sample subject HR quantiles (train PID 0):
  43JW: {0.1: 73.45, 0.25: 77.83, 0.5: 86.72, 0.75: 96.73, 0.9: 107.23}


## Feature Extraction

For each label timestamp, extract a 3-minute window. Compute statistics for raw, z-scored, and percentile-rank versions of every sensor. Add subject-baseline-deviation features comparing window mean to that subject's p10/p50/p90 baselines (encoding "how much above this subject's typical low/median/high levels is this window").

In [5]:
WINDOW_MS = 180_000
HALF_MS   = 90_000

ALL_FEATURE_COLS = SENSOR_COLS + Z_COLS + PR_COLS

def extract_features(label_df, sensor_df, refs):
    sensor_by_pid = {pid: grp.sort_values('timestamp').reset_index(drop=True)
                     for pid, grp in sensor_df.groupby('pid')}
    rows = []
    for n, lrow in enumerate(label_df.itertuples(index=False), 1):
        pid = lrow.pid; ts = float(lrow.timestamp); lid = int(lrow.id)
        feat = {'id': lid}
        sg = sensor_by_pid.get(pid)
        if sg is None:
            rows.append(feat); continue
        ta = sg['timestamp'].values
        mask_full  = (ta >= ts - WINDOW_MS) & (ta <= ts)
        mask_first = (ta >= ts - WINDOW_MS) & (ta < ts - HALF_MS)
        mask_last  = (ta >= ts - HALF_MS) & (ta <= ts)
        wa = sg.loc[mask_full,  ALL_FEATURE_COLS]
        wf = sg.loc[mask_first, ALL_FEATURE_COLS]
        wl = sg.loc[mask_last,  ALL_FEATURE_COLS]
        feat['window_count'] = int(len(wa))
        
        for c in ALL_FEATURE_COLS:
            v = wa[c].dropna().values.astype(float)
            vf = wf[c].dropna().values.astype(float)
            vl = wl[c].dropna().values.astype(float)
            if len(v) == 0:
                for s in ['mean','std','min','max','median','q25','q75','iqr','skew','delta']:
                    feat[f'{c}_{s}'] = np.nan
                continue
            feat[f'{c}_mean']   = float(np.mean(v))
            feat[f'{c}_std']    = float(np.std(v))
            feat[f'{c}_min']    = float(np.min(v))
            feat[f'{c}_max']    = float(np.max(v))
            feat[f'{c}_median'] = float(np.median(v))
            q25 = float(np.percentile(v, 25)); q75 = float(np.percentile(v, 75))
            feat[f'{c}_q25'] = q25; feat[f'{c}_q75'] = q75; feat[f'{c}_iqr'] = q75 - q25
            feat[f'{c}_skew']  = float(spstats.skew(v)) if len(v) > 2 else 0.0
            feat[f'{c}_delta'] = float(np.mean(vl) - np.mean(vf)) if len(vf) and len(vl) else 0.0
        
        # Subject-baseline-deviation features (raw sensors compared to subject quantiles)
        ref = refs.get(pid, {})
        for c in SENSOR_COLS:
            v = wa[c].dropna().values.astype(float)
            if len(v) == 0 or c not in ref:
                feat[f'{c}_dev_p10'] = feat[f'{c}_dev_p50'] = feat[f'{c}_dev_p90'] = np.nan
                feat[f'{c}_above_p75'] = feat[f'{c}_below_p25'] = np.nan
                continue
            mean_v = float(np.mean(v))
            feat[f'{c}_dev_p10'] = mean_v - ref[c][0.1]
            feat[f'{c}_dev_p50'] = mean_v - ref[c][0.5]
            feat[f'{c}_dev_p90'] = mean_v - ref[c][0.9]
            # Fraction of window samples above subject's p75 / below p25
            feat[f'{c}_above_p75'] = float(np.mean(v > ref[c][0.75]))
            feat[f'{c}_below_p25'] = float(np.mean(v < ref[c][0.25]))
        
        # Accel magnitude (raw and z-scored)
        ax, ay, az = wa['accel_x'].values, wa['accel_y'].values, wa['accel_z'].values
        if len(ax):
            mag = np.sqrt(ax**2 + ay**2 + az**2)
            feat['accel_mag_mean'] = float(np.mean(mag))
            feat['accel_mag_std']  = float(np.std(mag))
            feat['accel_mag_max']  = float(np.max(mag))
            axz, ayz, azz = wa['accel_x_z'].values, wa['accel_y_z'].values, wa['accel_z_z'].values
            magz = np.sqrt(axz**2 + ayz**2 + azz**2)
            feat['accel_magz_mean'] = float(np.mean(magz))
            feat['accel_magz_std']  = float(np.std(magz))
        else:
            for k in ['accel_mag_mean','accel_mag_std','accel_mag_max','accel_magz_mean','accel_magz_std']:
                feat[k] = np.nan
        
        # HRV-like
        hr = wa['heart_rate'].dropna().values
        if len(hr) >= 10:
            rr = 60000.0 / np.clip(hr, 30, 220)
            feat['hrv_sdnn']   = float(np.std(rr))
            feat['hrv_rmssd']  = float(np.sqrt(np.mean(np.diff(rr)**2))) if len(rr) > 1 else 0.0
            feat['hrv_meanrr'] = float(np.mean(rr))
        else:
            feat['hrv_sdnn'] = feat['hrv_rmssd'] = feat['hrv_meanrr'] = np.nan
        
        # Cross-sensor correlations (subject-relative magnitude relationships)
        try:
            hr_z = wa['heart_rate_z'].dropna().values
            eda_z = wa['eda_z'].dropna().values
            if len(hr_z) > 5 and len(eda_z) > 5 and len(hr_z) == len(eda_z):
                feat['corr_hr_eda'] = float(np.corrcoef(hr_z, eda_z)[0, 1])
            else:
                feat['corr_hr_eda'] = 0.0
        except Exception:
            feat['corr_hr_eda'] = 0.0
        
        rows.append(feat)
        if n % 200 == 0:
            print(f'  {n}/{len(label_df)}')
    return pd.DataFrame(rows).set_index('id')

print('Extracting train features...')
train_features = extract_features(TRAIN_LABEL, TRAIN_DATA, TRAIN_REFS)
print('Extracting test features...')
test_features  = extract_features(TEST_LABEL,  TEST_DATA,  TEST_REFS)
print('train features:', train_features.shape, 'test features:', test_features.shape)


Extracting train features...
  200/815
  400/815
  600/815
  800/815
Extracting test features...
  200/1028
  400/1028
  600/1028
  800/1028
  1000/1028
train features: (815, 220) test features: (1028, 220)


In [6]:
tli = TRAIN_LABEL.set_index('id')
y      = tli.loc[train_features.index, 'stress'].astype(int)
groups = tli.loc[train_features.index, 'pid'].astype(str)

common_cols = [c for c in train_features.columns if c in test_features.columns]
train_features = train_features[common_cols]
test_features  = test_features[common_cols]

imputer = SimpleImputer(strategy='median')
X      = pd.DataFrame(imputer.fit_transform(train_features), columns=common_cols, index=train_features.index)
X_test = pd.DataFrame(imputer.transform(test_features),       columns=common_cols, index=test_features.index)

# For sklearn models that benefit from scaling
scaler = StandardScaler()
X_scaled      = pd.DataFrame(scaler.fit_transform(X),      columns=X.columns,      index=X.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test),     columns=X_test.columns, index=X_test.index)

print('X     :', X.shape)
print('X_test:', X_test.shape)
print('y dist:', dict(Counter(y)))
print('Groups:', sorted(groups.unique()))


X     : (815, 220)
X_test: (1028, 220)
y dist: {1: 66, 0: 162, 2: 587}
Groups: ['43JW', 'C8Q6', 'DT5C', 'F1ZM', 'HDS9', 'P4DZ', 'TPQI']


## Honest CV — Heterogeneous Ensemble

GroupKFold by PID. Three model families:
- **LightGBM** (gradient boosting, axis-aligned splits)
- **HistGradientBoosting** (sklearn, also gradient boosting but different impl + missing-value handling)
- **ExtraTrees** (extremely randomized trees — high variance, low bias, very different inductive bias)

Each fold trains all three on the same fold and averages soft probabilities. Reported CV BA is on the averaged probabilities.

In [7]:
# Stronger regularization for cross-subject generalization
LGBM_PARAMS = dict(
    n_estimators=2000,
    learning_rate=0.02,
    num_leaves=15,
    max_depth=5,
    min_child_samples=30,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.6,
    reg_alpha=0.5,
    reg_lambda=1.0,
    class_weight='balanced',
    objective='multiclass',
    num_class=3,
    n_jobs=-1,
    verbose=-1,
)

HGBM_PARAMS = dict(
    learning_rate=0.05,
    max_iter=300,
    max_depth=6,
    max_leaf_nodes=15,
    min_samples_leaf=20,
    l2_regularization=1.0,
    class_weight='balanced',
    early_stopping=False,  # Disabled: stratified split fails when a CV fold has <2 of any class
)

ET_PARAMS = dict(
    n_estimators=600,
    max_depth=12,
    min_samples_leaf=5,
    max_features='sqrt',
    class_weight='balanced',
    n_jobs=-1,
    bootstrap=False,
)

unique_pids = sorted(groups.unique())
n_pids = len(unique_pids)
n_train = len(X)

def fit_lgbm(Xtr, ytr, Xva, yva, seed):
    m = lgb.LGBMClassifier(**{**LGBM_PARAMS, 'random_state': seed})
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)],
          callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)])
    return m

def fit_hgbm(Xtr, ytr, seed):
    m = HistGradientBoostingClassifier(**{**HGBM_PARAMS, 'random_state': seed})
    m.fit(Xtr, ytr)
    return m

def fit_et(Xtr, ytr, seed):
    m = ExtraTreesClassifier(**{**ET_PARAMS, 'random_state': seed})
    m.fit(Xtr, ytr)
    return m

SEEDS = [42, 7, 123]
oof_lgbm = np.zeros((n_train, 3))
oof_hgbm = np.zeros((n_train, 3))
oof_et   = np.zeros((n_train, 3))

print(f'GroupKFold splits = {n_pids} | seeds = {SEEDS}')
for seed in SEEDS:
    gkf = GroupKFold(n_splits=n_pids)
    for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups)):
        held_pid = groups.iloc[va_idx[0]]
        # LGBM uses raw features
        m1 = fit_lgbm(X.iloc[tr_idx], y.iloc[tr_idx], X.iloc[va_idx], y.iloc[va_idx], seed)
        oof_lgbm[va_idx] += m1.predict_proba(X.iloc[va_idx])
        # HGBM uses raw features (handles missing internally; we already imputed but fine)
        m2 = fit_hgbm(X.iloc[tr_idx], y.iloc[tr_idx], seed)
        oof_hgbm[va_idx] += m2.predict_proba(X.iloc[va_idx])
        # ExtraTrees uses scaled features
        m3 = fit_et(X_scaled.iloc[tr_idx], y.iloc[tr_idx], seed)
        oof_et[va_idx]   += m3.predict_proba(X_scaled.iloc[va_idx])
        if seed == SEEDS[0]:
            ba1 = balanced_accuracy_score(y.iloc[va_idx], m1.predict(X.iloc[va_idx]))
            ba2 = balanced_accuracy_score(y.iloc[va_idx], m2.predict(X.iloc[va_idx]))
            ba3 = balanced_accuracy_score(y.iloc[va_idx], m3.predict(X_scaled.iloc[va_idx]))
            print(f'  fold {fold} held-out {held_pid}: LGBM={ba1:.3f}, HGBM={ba2:.3f}, ET={ba3:.3f}')

oof_lgbm /= len(SEEDS); oof_hgbm /= len(SEEDS); oof_et /= len(SEEDS)

# Individual model OOF BAs
for name, oof in [('LGBM', oof_lgbm), ('HGBM', oof_hgbm), ('ExtraTrees', oof_et)]:
    ba = balanced_accuracy_score(y, oof.argmax(1))
    print(f'{name:12s} OOF BA: {ba:.4f}, dist: {dict(Counter(oof.argmax(1)))}')

# Find the best ensemble weighting via simple grid search on simplex
best = (-1.0, None, None)
oof_ensemble = None
for w1 in np.linspace(0, 1, 11):
    for w2 in np.linspace(0, 1 - w1, 11):
        w3 = 1 - w1 - w2
        if w3 < 0: continue
        e = w1 * oof_lgbm + w2 * oof_hgbm + w3 * oof_et
        ba = balanced_accuracy_score(y, e.argmax(1))
        if ba > best[0]:
            best = (ba, (w1, w2, w3), e)
print(f'\nBest ensemble weights (LGBM, HGBM, ET) = {tuple(round(w,2) for w in best[1])}, OOF BA = {best[0]:.4f}')
W_LGBM, W_HGBM, W_ET = best[1]
oof_ensemble = best[2]

print('\nEnsemble OOF distribution:', dict(Counter(oof_ensemble.argmax(1))))
print('Truth distribution        :', dict(Counter(y)))
print('Ensemble OOF confusion matrix:')
print(confusion_matrix(y, oof_ensemble.argmax(1)))
print('\nPer-PID OOF BA (ensemble):')
for pid in sorted(groups.unique()):
    mask = (groups == pid).values
    yt = y[mask]; yp = oof_ensemble[mask].argmax(1)
    if len(np.unique(yt)) >= 2:
        print(f'  {pid} (n={mask.sum()}, classes={dict(Counter(yt))}): BA={balanced_accuracy_score(yt, yp):.4f}')
    else:
        print(f'  {pid} (n={mask.sum()}, classes={dict(Counter(yt))}): acc={(yp==yt).mean():.4f}')


GroupKFold splits = 7 | seeds = [42, 7, 123]
  fold 0 held-out C8Q6: LGBM=0.500, HGBM=0.500, ET=0.461
  fold 1 held-out P4DZ: LGBM=0.391, HGBM=0.333, ET=0.333
  fold 2 held-out F1ZM: LGBM=0.481, HGBM=0.478, ET=0.451
  fold 3 held-out HDS9: LGBM=0.408, HGBM=0.524, ET=0.812
  fold 4 held-out 43JW: LGBM=0.192, HGBM=0.225, ET=0.187
  fold 5 held-out DT5C: LGBM=0.390, HGBM=0.345, ET=0.339
  fold 6 held-out TPQI: LGBM=0.343, HGBM=0.482, ET=0.435
LGBM         OOF BA: 0.3418, dist: {np.int64(1): 50, np.int64(2): 584, np.int64(0): 181}
HGBM         OOF BA: 0.2947, dist: {np.int64(2): 630, np.int64(0): 161, np.int64(1): 24}
ExtraTrees   OOF BA: 0.3147, dist: {np.int64(2): 661, np.int64(0): 129, np.int64(1): 25}

Best ensemble weights (LGBM, HGBM, ET) = (np.float64(1.0), np.float64(0.0), np.float64(0.0)), OOF BA = 0.3418

Ensemble OOF distribution: {np.int64(1): 50, np.int64(2): 584, np.int64(0): 181}
Truth distribution        : {1: 66, 0: 162, 2: 587}
Ensemble OOF confusion matrix:
[[ 32   8 122

## Calibration

Single knob: `proba × train_prior^alpha`, normalize. Tune alpha to make the OOF prediction distribution best match train prior (which equals OOF truth distribution by construction). Apply chosen alpha to test.

Smaller alphas are expected here than v18 because the ensemble is less class-2-biased than a single LightGBM (heterogeneous models cover for each other's blind spots).

In [8]:
def calibrate(proba, alpha, prior):
    cal = proba * (prior ** alpha)
    return cal / cal.sum(1, keepdims=True)

train_prior = np.array([Counter(y)[i] / len(y) for i in range(3)])
print('Train prior:', train_prior.round(3).tolist())
print()

# Tune alpha to match OOF predicted distribution to train prior
print('=== alpha sweep on OOF ensemble ===')
print(f'{"alpha":>6} | {"pred dist (0/1/2)":>22} | {"frac (0/1/2)":>22} | dev | OOF BA')
candidates = []
for alpha in np.linspace(-0.5, 2.0, 26):
    cal = calibrate(oof_ensemble, alpha, train_prior)
    preds = cal.argmax(1)
    counts = np.bincount(preds, minlength=3)
    fracs = counts / len(preds)
    dev = np.abs(fracs - train_prior).max()
    ba = balanced_accuracy_score(y, preds)
    candidates.append((alpha, dev, ba, counts, fracs))

# Print a subset
for alpha, dev, ba, counts, fracs in candidates:
    if abs(round(alpha*4) - alpha*4) < 1e-9 and round(alpha*2) == alpha*2:  # 0.5 step
        flag = ''
        print(f'{alpha:>6.2f} | {str(counts.tolist()):>22} | {str(fracs.round(3).tolist()):>22} | {dev:.3f} | {ba:.4f}{flag}')

# Two strategies: (a) OOF-distribution-match (anchor to ground truth proportions),
# (b) OOF-BA-max (greedy on metric).
candidates.sort(key=lambda x: x[1])
ALPHA_DIST = candidates[0][0]
candidates.sort(key=lambda x: -x[2])
ALPHA_BA = candidates[0][0]
print()
print(f'Alpha minimizing OOF dist deviation: {ALPHA_DIST:.2f} (dev={[c for c in candidates if c[0]==ALPHA_DIST][0][1]:.3f}, BA={[c for c in candidates if c[0]==ALPHA_DIST][0][2]:.4f})')
print(f'Alpha maximizing OOF BA:             {ALPHA_BA:.2f}')

# Choose the dist-matching alpha by default (more robust to OOF noise),
# unless OOF BA at that alpha is dramatically worse than the BA-optimal alpha.
ba_at_dist  = [c for c in candidates if c[0]==ALPHA_DIST][0][2]
ba_at_bamax = [c for c in candidates if c[0]==ALPHA_BA][0][2]
if ba_at_bamax - ba_at_dist > 0.05:
    # BA-max alpha is meaningfully better; use a compromise.
    ALPHA_FINAL = (ALPHA_DIST + ALPHA_BA) / 2.0
    print(f'Using compromise alpha (BA gain >0.05): {ALPHA_FINAL:.2f}')
else:
    ALPHA_FINAL = ALPHA_DIST
    print(f'Using dist-matching alpha (robust): {ALPHA_FINAL:.2f}')

cal_oof = calibrate(oof_ensemble, ALPHA_FINAL, train_prior)
print('\nFinal OOF BA after calibration:', round(balanced_accuracy_score(y, cal_oof.argmax(1)), 4))
print('Final OOF distribution         :', dict(Counter(cal_oof.argmax(1))))
print('Truth distribution             :', dict(Counter(y)))
print('Final OOF confusion matrix:')
print(confusion_matrix(y, cal_oof.argmax(1)))


Train prior: [0.199, 0.081, 0.72]

=== alpha sweep on OOF ensemble ===
 alpha |      pred dist (0/1/2) |           frac (0/1/2) | dev | OOF BA
 -0.50 |         [99, 445, 271] |  [0.121, 0.546, 0.333] | 0.465 | 0.4873
  0.00 |         [181, 50, 584] |  [0.222, 0.061, 0.717] | 0.023 | 0.3418
  0.50 |           [10, 3, 802] |  [0.012, 0.004, 0.984] | 0.264 | 0.3338
  1.00 |            [0, 0, 815] |        [0.0, 0.0, 1.0] | 0.280 | 0.3333
  1.50 |            [0, 0, 815] |        [0.0, 0.0, 1.0] | 0.280 | 0.3333
  2.00 |            [0, 0, 815] |        [0.0, 0.0, 1.0] | 0.280 | 0.3333

Alpha minimizing OOF dist deviation: 0.00 (dev=0.023, BA=0.3418)
Alpha maximizing OOF BA:             -0.10
Using compromise alpha (BA gain >0.05): -0.05

Final OOF BA after calibration: 0.4418
Final OOF distribution         : {np.int64(1): 110, np.int64(0): 224, np.int64(2): 481}
Truth distribution             : {1: 66, 0: 162, 2: 587}
Final OOF confusion matrix:
[[ 48  21  93]
 [  5  28  33]
 [171  61 355]]

## Final Test Prediction

Train all three models on **full training data** with multiple seeds (CV is already done for reporting). The final ensemble averages soft probabilities using the OOF-tuned weights, then applies the OOF-tuned calibration.

In [9]:
# Train final ensemble on ALL training data with multiple seeds.
# This maximizes the data each model sees; CV was for honest reporting only.
print('=== Training final ensemble on full training data ===')

test_lgbm = np.zeros((len(X_test), 3))
test_hgbm = np.zeros((len(X_test), 3))
test_et   = np.zeros((len(X_test), 3))

# For LGBM, we need a validation set for early stopping. Use one held-out PID per seed.
# This trains on 6 PIDs and uses the 7th for early stopping. We rotate which PID is used.
import itertools
pid_rotator = itertools.cycle(unique_pids)

for seed in SEEDS:
    holdout_pid = next(pid_rotator)
    tr_mask = (groups != holdout_pid).values
    va_mask = (groups == holdout_pid).values
    
    m1 = fit_lgbm(X[tr_mask], y[tr_mask], X[va_mask], y[va_mask], seed)
    test_lgbm += m1.predict_proba(X_test)
    
    # HGBM has internal early stopping on a stratified split
    m2 = fit_hgbm(X, y, seed)
    test_hgbm += m2.predict_proba(X_test)
    
    # ExtraTrees on scaled features
    m3 = fit_et(X_scaled, y, seed)
    test_et += m3.predict_proba(X_test_scaled)
    
    print(f'  seed {seed} done (LGBM held out {holdout_pid})')

test_lgbm /= len(SEEDS); test_hgbm /= len(SEEDS); test_et /= len(SEEDS)

print('\nIndividual model raw test distributions:')
for name, p in [('LGBM', test_lgbm), ('HGBM', test_hgbm), ('ExtraTrees', test_et)]:
    print(f'  {name:12s}: {dict(Counter(p.argmax(1)))}')

# Apply ensemble weights chosen by OOF
test_ensemble = W_LGBM * test_lgbm + W_HGBM * test_hgbm + W_ET * test_et
print(f'\nEnsemble (w={tuple(round(w,2) for w in (W_LGBM, W_HGBM, W_ET))}) raw test distribution: {dict(Counter(test_ensemble.argmax(1)))}')

# Apply calibration
cal_test = calibrate(test_ensemble, ALPHA_FINAL, train_prior)
final_preds = cal_test.argmax(1).astype(int)

submission = pd.DataFrame({'id': TEST_LABEL['id'].values, 'stress': final_preds})
submission.to_csv('submission.csv', index=False)

# Also save unweighted-average ensemble version as a backup
test_uniform = (test_lgbm + test_hgbm + test_et) / 3.0
cal_uniform = calibrate(test_uniform, ALPHA_FINAL, train_prior)
pd.DataFrame({'id': TEST_LABEL['id'].values, 
              'stress': cal_uniform.argmax(1).astype(int)}).to_csv('submission_uniform_weights.csv', index=False)

print('\n=== FINAL ===')
print('Saved submission.csv')
print(f'Final test prediction dist  : {dict(Counter(final_preds))}')
print(f'Train prior                 : {train_prior.round(3).tolist()}')
test_fracs = np.bincount(final_preds, minlength=3) / len(final_preds)
print(f'Test fractions              : {test_fracs.round(3).tolist()}')
print(f'Max abs deviation from prior: {np.abs(test_fracs - train_prior).max():.3f}')
print()
print(submission.head(10))


=== Training final ensemble on full training data ===
  seed 42 done (LGBM held out 43JW)
  seed 7 done (LGBM held out C8Q6)
  seed 123 done (LGBM held out DT5C)

Individual model raw test distributions:
  LGBM        : {np.int64(1): 68, np.int64(2): 351, np.int64(0): 609}
  HGBM        : {np.int64(2): 550, np.int64(0): 444, np.int64(1): 34}
  ExtraTrees  : {np.int64(2): 498, np.int64(0): 480, np.int64(1): 50}

Ensemble (w=(np.float64(1.0), np.float64(0.0), np.float64(0.0))) raw test distribution: {np.int64(1): 68, np.int64(2): 351, np.int64(0): 609}

=== FINAL ===
Saved submission.csv
Final test prediction dist  : {np.int64(1): 82, np.int64(2): 303, np.int64(0): 643}
Train prior                 : [0.199, 0.081, 0.72]
Test fractions              : [0.625, 0.08, 0.295]
Max abs deviation from prior: 0.427

     id  stress
0  1227       1
1  1228       2
2  1229       2
3  1230       2
4  1231       0
5  1232       0
6  1233       0
7  1234       0
8  1235       0
9  1236       0


## Summary

In [10]:
# Backup CSVs at neighboring alphas for diagnostic comparison
for alpha_alt in [max(0.0, ALPHA_FINAL - 0.5), max(0.0, ALPHA_FINAL - 0.3), ALPHA_FINAL + 0.3, ALPHA_FINAL + 0.5]:
    cal_alt = calibrate(test_ensemble, alpha_alt, train_prior)
    preds_alt = cal_alt.argmax(1).astype(int)
    counts = np.bincount(preds_alt, minlength=3)
    pd.DataFrame({'id': TEST_LABEL['id'].values, 'stress': preds_alt}).to_csv(f'submission_alpha_{alpha_alt:.2f}.csv', index=False)
    print(f'  submission_alpha_{alpha_alt:.2f}.csv: dist={counts.tolist()}')

# Raw ensemble (no calibration) for diagnostic
raw_preds = test_ensemble.argmax(1).astype(int)
pd.DataFrame({'id': TEST_LABEL['id'].values, 'stress': raw_preds}).to_csv('submission_raw_ensemble.csv', index=False)
print(f'  submission_raw_ensemble.csv: dist={list(np.bincount(raw_preds, minlength=3))}')

print()
print('========== v19 SUMMARY ==========')
print(f'Ensemble weights (LGBM, HGBM, ET): {tuple(round(w,2) for w in (W_LGBM, W_HGBM, W_ET))}')
print(f'Selected calibration alpha       : {ALPHA_FINAL:.2f}')
print(f'GroupKFold OOF BA (calibrated)   : {balanced_accuracy_score(y, cal_oof.argmax(1)):.4f}')
print(f'Test prediction dist             : {dict(Counter(final_preds))}')
print(f'Train prior                      : {train_prior.round(3).tolist()}')
print('==================================')
print()
print('SUBMIT: submission.csv')


  submission_alpha_0.00.csv: dist=[609, 68, 351]
  submission_alpha_0.00.csv: dist=[609, 68, 351]
  submission_alpha_0.25.csv: dist=[402, 4, 622]
  submission_alpha_0.45.csv: dist=[188, 0, 840]
  submission_raw_ensemble.csv: dist=[np.int64(609), np.int64(68), np.int64(351)]

========== v19 SUMMARY ==========
Ensemble weights (LGBM, HGBM, ET): (np.float64(1.0), np.float64(0.0), np.float64(0.0))
Selected calibration alpha       : -0.05
GroupKFold OOF BA (calibrated)   : 0.4418
Test prediction dist             : {np.int64(1): 82, np.int64(2): 303, np.int64(0): 643}
Train prior                      : [0.199, 0.081, 0.72]

SUBMIT: submission.csv


In [11]:
import pandas as pd
from collections import Counter
for f in ['submission.csv', 'submission_alpha_-0.55.csv', 'submission_alpha_-0.35.csv',
          'submission_alpha_0.25.csv', 'submission_alpha_0.45.csv', 'submission_uniform_weights.csv',
          'submission_raw_ensemble.csv']:
    try:
        s = pd.read_csv(f)
        d = dict(Counter(s['stress']))
        total = len(s)
        fracs = {k: round(v/total, 3) for k, v in sorted(d.items())}
        print(f'{f:42s} dist={d}  fracs={fracs}')
    except FileNotFoundError:
        print(f'{f}: not found')

submission.csv                             dist={1: 82, 2: 303, 0: 643}  fracs={0: 0.625, 1: 0.08, 2: 0.295}
submission_alpha_-0.55.csv: not found
submission_alpha_-0.35.csv: not found
submission_alpha_0.25.csv                  dist={2: 622, 0: 402, 1: 4}  fracs={0: 0.391, 1: 0.004, 2: 0.605}
submission_alpha_0.45.csv                  dist={2: 840, 0: 188}  fracs={0: 0.183, 2: 0.817}
submission_uniform_weights.csv             dist={2: 494, 0: 486, 1: 48}  fracs={0: 0.473, 1: 0.047, 2: 0.481}
submission_raw_ensemble.csv                dist={1: 68, 2: 351, 0: 609}  fracs={0: 0.592, 1: 0.066, 2: 0.341}
